# Debate

Supplementary material: code, data, and analysis for the detective debate study.

```
code/      generation pipeline (configs, prompts, core, scripts)
data/      accusation results and the 68 study dialogues
results/   participant judgments (results.csv) and H1–H2b analysis notebooks
```

Run this notebook from the repository root after `pip install -r requirements.txt`.

In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display
from IPython.utils.capture import capture_output

assert Path("code").is_dir() and Path("data").is_dir(), "Start Jupyter from the repository root."


def run(*args, tail=5):
    """Run a repo script with this kernel's Python and show the last lines of its output."""
    result = subprocess.run([sys.executable, *args], capture_output=True, text=True)
    print("\n".join((result.stdout + result.stderr).splitlines()[-tail:]))
    result.check_returncode()

## 1. Data

Each file in `data/dialogues/` is one study dialogue. `results/results.csv` has one row per participant judgment; `dialogue_id` matches the dialogue file name and `adjudication` is the side (A or B) the participant chose.

In [ ]:
results = pd.read_csv("results/results.csv")
print(results.shape)
results.head()

In [ ]:
dialogue = json.loads(Path("data/dialogues/01.json").read_text())
print(dialogue["topic"], "\n")
for turn in dialogue["turns"]:
    print(f"[{turn['speaker']}] {turn['utterance']}\n")

## 2. Build detective cases

The detective cases are not included. Build them from the source dataset, [True Detective](https://github.com/MaksymDel/true-detective) (Del & Fishel, 2023): clone it with `git clone https://github.com/MaksymDel/true-detective data/true-detective`, then set `BUILD_CASES = True`. The converter writes the four study cases to `data/detective_cases/`. Its suspect evidence analysis calls an LLM through OpenRouter, so it needs `OPENROUTER_API_KEY`.

In [ ]:
BUILD_CASES = False

if BUILD_CASES:
    assert Path("data/true-detective/data/data.zip").exists(), "Clone True Detective into data/true-detective/ first."
    assert os.environ.get("OPENROUTER_API_KEY"), "Set OPENROUTER_API_KEY first."
    run("code/convert_true_detective.py", tail=1)

## 3. Generate dialogues

Requires the detective cases from step 2. Settings are in `code/configs/generation_detective.yaml`; outputs go to `outputs/`. A dry run prints the schedule without calling any API.

In [ ]:
if Path("data/detective_cases/_index.json").exists():
    run("code/scripts/run_detective_v2.py", "--step", "full", "--dry-run")
else:
    print("Skipped: build the detective cases in step 2 first.")

Real generation calls OpenRouter and needs `OPENROUTER_API_KEY`. Set `RUN_GENERATION = True` to run one case (`--smoke-test`), or drop that flag for the full sweep.

In [ ]:
RUN_GENERATION = False

if RUN_GENERATION:
    assert os.environ.get("OPENROUTER_API_KEY"), "Set OPENROUTER_API_KEY first."
    run("code/scripts/run_detective_v2.py", "--step", "full", "--smoke-test", tail=20)

## 4. Analysis

`results/primary_analysis_H1.ipynb`, `_H2a`, and `_H2b` test hypotheses H1–H2b on `results/results.csv`; open them for the full analysis. The cell below runs all three (about 2 minutes) and shows each one's final table. H2a also writes `results/power_curve.csv` and `results/power_curve.png`.

In [ ]:
def run_notebook(path):
    """Run every code cell of a notebook in a fresh namespace and return that namespace."""
    namespace = {}
    for cell in json.loads(Path(path).read_text())["cells"]:
        if cell["cell_type"] == "code":
            exec("".join(cell["source"]), namespace)
    return namespace


final_tables = {
    "H1": ("results/primary_analysis_H1.ipynb", "aggregate_effect_summary"),
    "H2a": ("results/primary_analysis_H2a.ipynb", "h2a_publication_table"),
    "H2b": ("results/primary_analysis_H2b.ipynb", "h2b_publication_table"),
}

for hypothesis, (path, table) in final_tables.items():
    with capture_output():
        namespace = run_notebook(path)
    print(hypothesis)
    display(namespace[table])